In [ ]:
import pandas as pd
import os
from pathlib import Path

# 1. Configurando os Caminhos (Padrão Cooper/Silver)
silver_path = Path("Data/Silver")
silver_path.mkdir(parents=True, exist_ok=True)

# 2. Carregando os dados (Garantindo que o df_serie_a existe)
# Se você já tiver o df_serie_a na memória, ele usa. Se não, tenta ler o Parquet.
if 'df_serie_a' not in locals():
    print("📂 Carregando dados da it Serie A...")
    df_serie_a = pd.read_parquet(silver_path / "serie_a_silver.parquet")

# 3. Definição das Métricas de "Aura"
# Essas métricas indicam a força ofensiva e controle do time
metrics = ['Gls', 'Ast', 'Sh', 'SoT', 'xG']

# 4. Cálculo da Média Móvel (Rolling Stats - Os últimos 5 jogos)
# O shift(1) é vital: ele garante que o modelo só veja o PASSADO.
print("🪄 Calculando a Aura (Médias Móveis)...")
df_serie_a = df_serie_a.sort_values(['Squad', 'Date'])
rolling_stats = df_serie_a.groupby('Squad')[metrics].rolling(window=5, min_periods=1).mean().shift(1)

# Resetar o índice para bater com o DataFrame original
rolling_stats = rolling_stats.reset_index(level=0, drop=True)
df_serie_a[[f'{m}_rolling5' for m in metrics]] = rolling_stats

# 5. Salvando a Tabela "Lapidada" (Pronta para a Gold Layer)
df_aura_final = df_serie_a.dropna(subset=['Gls_rolling5'])
output_file = silver_path / "serie_a_aura_final.parquet"
df_aura_final.to_parquet(output_file, index=False)

print(f"✅ Aura calculada e salva em: {output_file}")
print(f"📊 Total de registros prontos: {len(df_aura_final)}")